# TFG Roberto García García

* Importamos las librerías básicas de Python para manejar datos y archivos, y luego recorremos el directorio donde se encuentran los datos mostrando la ruta completa de cada archivo que encuentra.

In [ ]:
# Importamos las librerías principales para el análisis de datos y la manipulación de archivos.
import numpy as np
import pandas as pd
import os
import sys

# Mostramos los archivos disponibles en el directorio de datos local
data_dir = r"C:\Users\rober\Desktop\Universidad\TFG\DATASETS"  # Creamos la ruta local
for dirname, _, filenames in os.walk(data_dir):
    for filename in filenames:
        print(os.path.join(dirname, filename))

C:\Users\rober\Desktop\Universidad\TFG\DATASETS\UNSW-NB15_1.csv
C:\Users\rober\Desktop\Universidad\TFG\DATASETS\UNSW-NB15_2.csv
C:\Users\rober\Desktop\Universidad\TFG\DATASETS\UNSW-NB15_3.csv
C:\Users\rober\Desktop\Universidad\TFG\DATASETS\UNSW-NB15_4.csv
C:\Users\rober\Desktop\Universidad\TFG\DATASETS\UNSW-NB15_features.csv
C:\Users\rober\Desktop\Universidad\TFG\DATASETS\UNSW-NB15_LIST_EVENTS.csv
C:\Users\rober\Desktop\Universidad\TFG\DATASETS\UNSW_NB15_testing-set.csv
C:\Users\rober\Desktop\Universidad\TFG\DATASETS\UNSW_NB15_training-set.csv


* Indicamos la reproducibilid y mostramos las versiones de Python, NumPy y Pandas.

In [3]:
# Reproducibilidad
SEED = 42
np.random.seed(SEED)

# Versiones de las librerías
print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
NumPy: 1.26.4
Pandas: 1.5.3


* Definimos las rutas locales de los archivos, el nombre de las columnas objetivo, listas de columnas que no se usan como características y variables categóricas que se usarán como características.

In [ ]:
# Definimos las rutas locales de los archivos de datos
TRAIN_PATH = "C:\\Users\\rober\\Desktop\\Universidad\\TFG\\DATASETS\\UNSW_NB15_training-set.csv"    # Ruta del conjunto de entrenamiento
TEST_PATH  = "C:\\Users\\rober\\Desktop\\Universidad\\TFG\\DATASETS\\UNSW_NB15_testing-set.csv"     # Ruta del conjunto de prueba
FEAT_PATH  = "C:\\Users\\rober\\Desktop\\Universidad\\TFG\\DATASETS\\NUSW-NB15_features.csv"        # Ruta del archivo con la descripción de las características

# Columnas objetivo para la clasificación binaria y multiclase
TARGET_BINARY = "label"         # Etiqueta binaria: 0 (normal) o 1 (ataque)
TARGET_MULTI  = "attack_cat"    # Etiqueta multiclase: tipo de ataque (varios valores posibles)

# Columnas excluidas como características (no se utilizan para el entrenamiento del modelo)
NON_FEATURE_COLS = ["id", TARGET_BINARY, TARGET_MULTI]      # Columnas que no se consideran características para el modelo
IP_COLS = ["srcip", "dstip"]                                # Columnas de direcciones IP que no se utilizan como características

# Columnas categóricas que se utilizarán en el análisis
CATEGORICAL_FEATURES = ["proto", "service", "state"]

# Resumen de las constantes definidas
print("Constantes definidas ✅")
print("Targets:", TARGET_BINARY, TARGET_MULTI)
print("Categoricals:", CATEGORICAL_FEATURES)


Constants set ✅
Targets: label attack_cat
Categoricals: ['proto', 'service', 'state']


* Preparamos funciones reutilizables para evaluar los modelos de clasificación del notebook.

In [5]:
# Importamos las métricas de evaluación del modelo desde la librería scikit-learn.
from sklearn.metrics import (
    accuracy_score,         # Precisión del modelo
    f1_score,               # Puntuación F1 del modelo
    recall_score,           # Sensibilidad del modelo
    classification_report,  # Informe de clasificación
    confusion_matrix,       # Matriz de confusión
)

# Definimos una función para evaluar un modelo binario
def evaluate_binary(model, X_test, y_test):
    """
    Returns que proporciona la función:
      - accuracy (precisión)
      - macro_f1 (promedio de la puntuación F1)
      - attack_recall (recall de la clase positiva, pos_label=1)
    """
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
        "attack_recall": recall_score(y_test, y_pred, pos_label=1),
    }

# Definimos una función para evaluar un modelo multiclase
def evaluate_multiclass(model, X_test, y_test):
    """
    Returns que proporciona la función:
      - accuracy (precisión)
      - macro_f1 (promedio de la puntuación F1)
    """
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
    }

# Definimos una función para mostrar un informe de clasificación
def report_multiclass(y_true, y_pred):
    """
    Devuelve un diccionario y una tabla con el informe
    """
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    df = pd.DataFrame(rep).T.sort_values("f1-score", ascending=False)
    return rep, df

* Definimos funciones auxiliares para dibujar matrices de confusión y gráficas de recall por clase. De este modo visualizamos como funciona el modelo en problemas de clasificación.

In [ ]:
# Importamos la librería matplotlib para la visualización de datos y configuramos el estilo de los gráficos.
import matplotlib.pyplot as plt

# Definimos una función para representar la matriz de confusión
def plot_confusion_matrix(cm, labels, title):
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

# Definimos una función para representar el recall por clase
def plot_per_class_recall(report_df, title):
    """
    report_df: DataFrame con el resultado del informe especificado.
    Expects rows for classes plus macro avg/weighted avg/accuracy.
    """
    df = report_df.copy()
    df = df[~df.index.isin(["accuracy", "macro avg", "weighted avg"])]
    df = df.sort_values("recall", ascending=False)

    plt.figure(figsize=(10, 5))
    plt.bar(df.index, df["recall"])
    plt.title(title)
    plt.ylabel("Recall")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

* Cargamos los datos crudos de entrenamiento, prueba y el archivo de metadatos de características. Comprobamos que los conjuntos tengan la misma estructura.

In [7]:
# Definimos los conjuntos de entrenamiento, prueba y metadatos de características
train_raw = pd.read_csv("C:\\Users\\rober\\Desktop\\Universidad\\TFG\\DATASETS\\UNSW_NB15_training-set.csv")
test_raw  = pd.read_csv("C:\\Users\\rober\\Desktop\\Universidad\\TFG\\DATASETS\\UNSW_NB15_testing-set.csv")
feat_meta = pd.read_csv("C:\\Users\\rober\\Desktop\\Universidad\\TFG\\DATASETS\\UNSW-NB15_features.csv", encoding="latin-1")

# Mostramos filas y columnas de los conjuntos de datos
print("TRAIN_RAW shape:", train_raw.shape)
print("TEST_RAW  shape:", test_raw.shape)
print("FEAT_META shape:", feat_meta.shape)

# Comprobamos que ambos conjuntos tengan el mismo número de columnas
assert train_raw.shape[1] == test_raw.shape[1], "Column count differs between splits!"
print("✅ Mismo número de columnas en ambos conjuntos de datos.")

TRAIN_RAW shape: (82332, 45)
TEST_RAW  shape: (175341, 45)
FEAT_META shape: (49, 4)
✅ Mismo número de columnas en ambos conjuntos de datos.


* Comparamos las columnas del conjunto de entrenamiento y el conjunto de prueba para comprobar que tengan el mismo orden y las mismas variables. En caso de haber diferencias, las mostramos para detectar posibles inconsistencias.

In [8]:
# Convertimos los nombres de las columnas a una lista
train_cols = list(train_raw.columns)
test_cols  = list(test_raw.columns)

# Comprobamos que ambos conjuntos tengan el mismo orden de columnas
same_order = (train_cols == test_cols)
same_set = (set(train_cols) == set(test_cols))

# Mostramos los resultados de la comprobación
print("Mismo orden de columnas:", same_order)
print("Mismo conjunto de columnas:", same_set)

# En caso de que el orden no coincida, mostramos las diferencias
if not same_order:
    
    diffs = [(i, train_cols[i], test_cols[i]) for i in range(len(train_cols)) if train_cols[i] != test_cols[i]]
    print("\nPrimeras 10 diferencias de orden (index, train_col, test_col):")
    for d in diffs[:10]:
        print(d)

if not same_set:
    print("\nColumnas solo en TRAIN_RAW:", sorted(set(train_cols) - set(test_cols)))
    print("Columnas solo en TEST_RAW :", sorted(set(test_cols) - set(train_cols)))


# Imprimimos las columnas ordenadas
print("\nColumnas ordenadas:")
for i, col in enumerate(train_cols):
    print(f"{i:02d}: {col}")

Mismo orden de columnas: True
Mismo conjunto de columnas: True

Columnas ordenadas:
00: id
01: dur
02: proto
03: service
04: state
05: spkts
06: dpkts
07: sbytes
08: dbytes
09: rate
10: sttl
11: dttl
12: sload
13: dload
14: sloss
15: dloss
16: sinpkt
17: dinpkt
18: sjit
19: djit
20: swin
21: stcpb
22: dtcpb
23: dwin
24: tcprtt
25: synack
26: ackdat
27: smean
28: dmean
29: trans_depth
30: response_body_len
31: ct_srv_src
32: ct_state_ttl
33: ct_dst_ltm
34: ct_src_dport_ltm
35: ct_dst_sport_ltm
36: ct_dst_src_ltm
37: is_ftp_login
38: ct_ftp_cmd
39: ct_flw_http_mthd
40: ct_src_ltm
41: ct_srv_dst
42: is_sm_ips_ports
43: attack_cat
44: label


* Comparamos los tipos de datos de cada columna en el conjunto de entrenamiento y de prueba. De esta forma comprobamos que las columnas tengan el mismo tipo.

In [9]:
# Obtenemos los tipos de datos de las columnas
dtype_train = train_raw.dtypes.astype(str)
dtype_test  = test_raw.dtypes.astype(str)

# Creamos un DataFrame para comparar los resultados
dtype_cmp = pd.DataFrame({
    "dtype_train": dtype_train,
    "dtype_test": dtype_test,
})
dtype_cmp["same"] = (dtype_cmp["dtype_train"] == dtype_cmp["dtype_test"])

# Filtramos las columnas donde los tipos no coinciden
mismatched = dtype_cmp[~dtype_cmp["same"]]

# Mostramos las columnas con tipos de datos diferentes
print("Columnas con tipos diferentes:", len(mismatched))
if len(mismatched) > 0:
    display(mismatched)
else:
    print("✅ Todos los tipos de datos coinciden entre TRAIN_RAW y TEST_RAW.")

Columnas con tipos diferentes: 0
✅ Todos los tipos de datos coinciden entre TRAIN_RAW y TEST_RAW.


* Comprobamos que las columnas objetivo estén presentes tanto en el conjunto de entrenamiento como en el conjunto de prueba. Además, se inspeccionan los valores de esas etiquetas para ver si coinciden entre ambos conjuntos.

In [10]:
# Definimos los nombres de las columnas de etiqueta binaria y multiclase
TARGET_BINARY = "label"
TARGET_MULTI  = "attack_cat"

# Comprobamos que las columnas de etiquetas existen en ambos conjuntos de datos
for col in [TARGET_BINARY, TARGET_MULTI]:
    assert col in train_raw.columns, f"Columna {col} no encontrada en train_raw"
    assert col in test_raw.columns, f"Columna {col} no encontrada en test_raw"

# Valores de la etiqueta binaria
train_bin_vals = sorted(train_raw[TARGET_BINARY].dropna().unique().tolist())
test_bin_vals  = sorted(test_raw[TARGET_BINARY].dropna().unique().tolist())

print("Valores de la etiqueta binaria en TRAIN_RAW:", train_bin_vals)
print("Valores de la etiqueta binaria en TEST_RAW :", test_bin_vals)

# Valores de la etiqueta multiclase
train_mc_vals = sorted(train_raw[TARGET_MULTI].dropna().unique().tolist())
test_mc_vals  = sorted(test_raw[TARGET_MULTI].dropna().unique().tolist())

print("\nClases multiclase en TRAIN_RAW (n=%d):" % len(train_mc_vals), train_mc_vals)
print("Clases multiclase en TEST_RAW  (n=%d):" % len(test_mc_vals), test_mc_vals)

# Calculamos que clases aparecen solo en TRAIN_RAW y solo en TEST_RAW
only_in_train = sorted(set(train_mc_vals) - set(test_mc_vals))
only_in_test  = sorted(set(test_mc_vals) - set(train_mc_vals))

print("\nClases solo en TRAIN_RAW:", only_in_train if only_in_train else "None")
print("Clases solo en TEST_RAW :", only_in_test if only_in_test else "None")

Valores de la etiqueta binaria en TRAIN_RAW: [0, 1]
Valores de la etiqueta binaria en TEST_RAW : [0, 1]

Clases multiclase en TRAIN_RAW (n=10): ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
Clases multiclase en TEST_RAW  (n=10): ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']

Clases solo en TRAIN_RAW: None
Clases solo en TEST_RAW : None


* Celda 10